# 04 - PHASE 1 GATE: reliability and predictability of delta_y

AGENTS.md Sec 6 - the most important experiment in the repo. Cheap, and if negative it saves
months. **Nothing under pcc/method/ may be built until this passes.**

- **Gate A (Sec 6.2):** split-half reliability of delta_y >= 0.30. This is the CEILING on any
  achievable R2; reporting R2 without it is an analysis error.
- **Gate B (Sec 6.3):** held-out-CLASS R2 clearly > 0, CI excluding 0, normalized by ceilings.
- **Gate C (Sec 6.3):** geometry must beat log-prevalence-only and nearest-class-distance-only.
  Sec 6.5: if log-prevalence explains most of delta_y, the space is taken and the project stops.
- **Sec 6.4:** predicted delta must actually SHRINK sets on held-out classes.

## Amendment 2 (reports/protocol_amendments.md) is applied here

delta_y is estimated at **MATCHED n_cal per class**. The quantile estimator's bias depends on
group size, and n_y is proportional to prevalence, so unmatched delta_y correlates MECHANICALLY
with prevalence even with zero class structure (measured corr -0.385 -> +0.016 after matching).
Without this, gate C could kill the project on an estimator artefact. A **permutation null**
(labels shuffled, class sizes preserved) calibrates what prevalence 'predicts' under no
structure; conclusions must exceed the null, not merely exceed zero.

## Descriptor policy (pre-registered, reports/descriptor_stability_findings.md)

**PRIMARY = `stable`** (features with cross-draw stability >= 0.90 at the chosen quota),
**SECONDARY = `full`**. Both always reported. The selection criterion never touches delta_y,
so it cannot bias the outcome.

## CIFAR-100 LIMITS - what this run can and cannot decide

1. Only **alpha=0.1** is feasible (100 test images/class; see phase0_checkpoint_gate.md).
2. Gate A is **noise-limited**: split-half leaves ~25 samples/class, so a low reliability here
   is uninterpretable.
3. CIFAR-100 is **perfectly balanced** (500 train images/class), so `log_prevalence` has NO
   cross-class variance -> **gate C's prevalence ablation is UNDEFINED here.** Only the
   distance ablation is testable. Gate C must be decided on Pl@ntNet.
4. 100 classes -> 50 fit / 50 held-out: too thin for the extrapolation claim.

**So this notebook DEBUGS the Phase-1 machinery. It does not decide the gate.**


## 1. Config - `# === EDIT ME ===`


In [ ]:
# === EDIT ME ===========================================================
REPO_URL   = ''
REPO_DIR   = 'foundation-cp'
DRIVE_ROOT = '/content/drive/MyDrive/pcc'
DATASET    = 'cifar100'
BACKBONE   = 'resnet50_self'

ALPHA      = 0.1        # only feasible alpha at 100 test images/class
N_CAL      = 25         # matched calibration samples per class (Amendment 2)
N_SPLITS_A = 100        # split-half repetitions for gate A (Sec 6.2)
N_SPLITS_BC= 100        # class-level splits for gate B/C (Sec 6.3)
N_NULL     = 50         # permutation-null repetitions (Amendment 2)
STABLE_THRESHOLD = 0.90 # descriptor stability cut for the PRIMARY feature set
SEED = 42
EMB_DIR = f'{DRIVE_ROOT}/embeddings/{DATASET}/{BACKBONE}'
# =======================================================================
print('EMB_DIR =', EMB_DIR, '| alpha', ALPHA, '| matched n_cal', N_CAL)


## 2. Mount Drive + repo + env + seed


In [ ]:
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')
if REPO_URL and not os.path.isdir(REPO_DIR):
    subprocess.run(['git','clone',REPO_URL,REPO_DIR], check=True)
if os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR if os.path.isabs(REPO_DIR) else '/content/'+REPO_DIR)
os.environ['PYTHONPATH'] = os.getcwd() + os.pathsep + os.environ.get('PYTHONPATH','')
subprocess.run(['pip','install','-q','-r','requirements.txt'], check=False)
from pcc.utils.seed import set_seed; from pcc.utils.io import environment_stamp
set_seed(SEED)
print('env:', environment_stamp()['packages'])


## 3. Load data. Scores from TEST (cal/eval); descriptor embeddings from TRAIN only (Sec 6.3)


In [ ]:
import numpy as np, os
te = np.load(os.path.join(EMB_DIR, 'test.npz'))
tr = np.load(os.path.join(EMB_DIR, 'train_subset.npz'))
logits, labels = te['logits'], te['labels']
tr_emb, tr_lab, tr_lg = tr['embeddings'], tr['labels'], tr['logits']
n, K = logits.shape
print(f'test scores {logits.shape} | train embeddings {tr_emb.shape}')
print(f'test samples/class ~{n//K} | train images/class ~{len(tr_lab)//K}')

# LEAK GUARD (Sec 8.3): descriptors come from TRAIN, conformal scores from TEST.
# These are disjoint datasets by construction; assert the class sets line up.
assert set(np.unique(tr_lab)) <= set(range(K)), 'train labels outside class range'
print('descriptor source = TRAIN split; calibration/eval = TEST split (disjoint)')


## 4. GATE A - split-half reliability of delta_y (Sec 6.2). Sets the R2 ceiling.


In [ ]:
from pcc.scores.base import thr_lac
from pcc.targets.delta import split_half_reliability
from pcc.eval.stats import mean_ci
import numpy as np

# THR/LAC scores from the softmax of the test logits
e = np.exp(logits - logits.max(1, keepdims=True))
probs = e / e.sum(1, keepdims=True)
S = thr_lac(probs)
s_true = S[np.arange(n), labels]

relA = split_half_reliability(s_true, labels, K, ALPHA,
                              n_splits=N_SPLITS_A, seed=SEED)
rel_ci = mean_ci(relA['reliability_splits'][~np.isnan(relA['reliability_splits'])])
r_delta = relA['reliability_mean']
print(f"gate A reliability (Spearman-Brown) = {r_delta:.3f} "
      f"95% CI [{rel_ci['ci_low']:.3f}, {rel_ci['ci_high']:.3f}]  (threshold 0.30)")
print('gate A:', 'PASS' if rel_ci['ci_low'] >= 0.30 else 'FAIL/INCONCLUSIVE')
print('NOTE: ~25 samples/class after halving -> noise-limited on CIFAR-100.')


## 5. delta_y at MATCHED n_cal + prevalence null (Amendment 2)


In [ ]:
from pcc.targets.delta import delta_y, delta_y_matched_n, prevalence_null
import numpy as np

counts_test = np.bincount(labels, minlength=K)
d_unmatched = delta_y(s_true, labels, K, ALPHA)
d_matched, kept = delta_y_matched_n(s_true, labels, K, ALPHA,
                                    n_cal=N_CAL, seed=SEED)
print(f'unmatched delta_y defined for {np.isfinite(d_unmatched).sum()}/{K} classes')
print(f'matched   delta_y defined for {int((kept & np.isfinite(d_matched)).sum())}/{K} classes'
      f' (n_cal={N_CAL})')

nullres = prevalence_null(s_true, labels, K, ALPHA, n_cal=N_CAL,
                          n_reps=N_NULL, seed=SEED)
if nullres['n_reps'] == 0:
    print('prevalence null UNDEFINED:', nullres.get('undefined_reason'))
else:
    print('prevalence null: mean %.3f sd %.3f |null|p95 %.3f (n=%d)' % (
          nullres['null_mean'], nullres['null_sd'],
          nullres['null_abs_p95'], nullres['n_reps']))
print('CIFAR-100 is BALANCED, so prevalence has no cross-class variance here ->')
print('the prevalence ablation is UNDEFINED on this dataset (see cell 0, limit 3).')

delta_primary = np.where(kept, d_matched, np.nan)


## 6. Descriptors phi(y) from TRAIN data, and the two pre-registered feature sets


In [ ]:
from pcc.descriptors.phi import build_descriptors
from pcc.descriptors.stability import descriptor_stability, QUOTA_DETERMINED
import numpy as np

# full train counts drive log_prevalence (NOT the extraction quota)
train_counts = np.full(K, 500)   # CIFAR-100 has 500 train images per class
Phi, names = build_descriptors(tr_emb, tr_lg, tr_lab, K,
                              log_prevalence_from=train_counts)
print(f'Phi {Phi.shape} features={len(names)}')

stab = descriptor_stability(tr_emb, tr_lg, tr_lab, K, quotas=(100,), n_reps=3,
                            seed=SEED, stable_threshold=STABLE_THRESHOLD)
per_feat = stab['by_quota'][100]['per_feature']
stable_names = [nm for nm in names
                if nm not in QUOTA_DETERMINED
                and np.isfinite(per_feat.get(nm, np.nan))
                and per_feat[nm] >= STABLE_THRESHOLD]
print('PRIMARY (stable) features:', stable_names)
print('dropped as unstable  :', [nm for nm in names
      if nm not in stable_names and nm not in QUOTA_DETERMINED])
FEATURE_SETS = {'stable': stable_names, 'full': [nm for nm in names]}


## 7. GATE B/C - class-level split predictability, normalized by BOTH ceilings

Reported R2 is divided by the target ceiling (gate A) AND the descriptor ceiling; a perfect
model cannot reach 1.0 when either is < 1 (descriptor_stability_findings.md).


In [ ]:
from pcc.eval.predictability import predictability
import numpy as np

r_phi = stab['by_quota'][100]['mean_corr']
print(f'ceilings: target r_delta={r_delta:.3f}  descriptor r_phi={r_phi:.3f}  '
      f'joint~{r_delta*r_phi:.3f}')

gateBC = {}
for setname, feats in FEATURE_SETS.items():
    cols = [names.index(f) for f in feats]
    res = predictability(Phi[:, cols], delta_primary, feats,
                         reliability=r_delta, n_splits=N_SPLITS_BC, seed=SEED)
    gateBC[setname] = res
    raw = res['r2_by_predictor']['full']
    print(f"[{setname}] n_classes={res['n_classes_used']} "
          f"held-out R2={raw['mean']:+.3f} CI [{raw['ci_low']:+.3f},{raw['ci_high']:+.3f}]")
    if res['normalized_full_r2']:
        nz = res['normalized_full_r2']
        print(f"           normalized by r_delta: {nz['mean']:+.3f} "
              f"CI [{nz['ci_low']:+.3f},{nz['ci_high']:+.3f}]")
    print(f"           gate B: {'PASS' if res['gate_B_pass'] else 'FAIL'}")
    for abl, det in res['gate_C_detail'].items():
        pd_ = det['paired_diff']
        print(f"           vs {abl:22s} diff={pd_['mean']:+.3f} "
              f"CI [{pd_['ci_low']:+.3f},{pd_['ci_high']:+.3f}] "
              f"beats={det['full_beats_it']}")


## 8. Sec 6.4 - does predicted delta actually SHRINK held-out-class sets?


In [ ]:
from pcc.eval.leakguard import class_level_split
from pcc.eval.predictability import ridge_fit, ridge_predict
from pcc.eval.setsize import setsize_translation_holdout
from pcc.eval.decomposition import group_quantile
import numpy as np

rng = np.random.default_rng(SEED)
usable = np.where(np.isfinite(delta_primary) & np.isfinite(Phi).all(axis=1))[0]
cols = [names.index(f) for f in FEATURE_SETS['stable']]
sizes = []
for rep in range(20):
    perm = rng.permutation(usable)
    fit_c, held_c = perm[:len(perm)//2], perm[len(perm)//2:]
    model = ridge_fit(Phi[fit_c][:, cols], delta_primary[fit_c], 1.0)
    dhat = np.full(K, 0.0)
    dhat[held_c] = ridge_predict(model, Phi[held_c][:, cols])
    # calibrate the global threshold on a sample split, evaluate on the rest
    idx = rng.permutation(n); cal, ev = idx[:n//2], idx[n//2:]
    qg = group_quantile(S[cal, labels[cal]], ALPHA, 'conformal')
    mask = np.isin(labels[ev], held_c)
    if mask.sum() == 0: continue
    res = setsize_translation_holdout(S[ev][mask], labels[ev][mask], K, ALPHA,
                                      qg, dhat, held_c)
    sizes.append(res['avg_set_size_delta'])
from pcc.eval.stats import mean_ci
sz = mean_ci(sizes)
print(f"Sec 6.4 held-out set-size change = {sz['mean']:+.4f} "
      f"95% CI [{sz['ci_low']:+.4f}, {sz['ci_high']:+.4f}]  (negative = SMALLER sets)")
print('Sec 6.4:', 'POSITIVE (sets shrink)' if sz['ci_high'] < 0 else 'NOT POSITIVE')


## 9. Write report - explicit verdict, and what CIFAR-100 cannot decide


In [ ]:
import time, numpy as np
from pcc.utils.io import write_report

def clean(o):
    if isinstance(o, dict): return {k: clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [clean(v) for v in o]
    if isinstance(o, np.ndarray): return None
    if isinstance(o, (np.floating, np.integer)): return float(o)
    return o

results = {'gate_A': {'reliability': float(r_delta),
                      'ci_low': float(rel_ci['ci_low']),
                      'ci_high': float(rel_ci['ci_high']),
                      'threshold': 0.30, 'noise_limited': True},
           'ceilings': {'r_delta': float(r_delta), 'r_phi': float(r_phi)},
           'matched_n_cal': N_CAL,
           'classes_with_delta': int((kept & np.isfinite(d_matched)).sum()),
           'prevalence_null': clean(nullres),
           'gate_BC': clean({k: {kk: vv for kk, vv in v.items()}
                             for k, v in gateBC.items()}),
           'sec_6_4': {'mean': float(sz['mean']), 'ci_low': float(sz['ci_low']),
                       'ci_high': float(sz['ci_high'])},
           'cifar100_cannot_decide': ['alpha=0.01 infeasible',
                                      'gate A noise-limited at ~25 samples/class',
                                      'balanced dataset -> prevalence ablation undefined',
                                      '100 classes -> 50/50 class split too thin'],
           'debug_only': True}

report = write_report('pcc/reports', f'04_phase1_gate_{DATASET}',
    hypothesis='delta_y is a reliable class-level signal (A) predictable from class geometry '
               '(B) beyond trivial predictors (C), and predicted delta shrinks held-out sets (6.4)',
    pass_criteria='A: split-half reliability CI low >= 0.30; B: held-out R2 CI excludes 0 after '
                  'normalizing by the target AND descriptor ceilings; C: full descriptor beats '
                  'log-prevalence-only and distance-only with paired-diff CI excluding 0 and '
                  'exceeding the permutation null; 6.4: held-out set size decreases with CI '
                  'below 0. delta_y estimated at matched n_cal (Amendment 2). CIFAR-100 is '
                  'DEBUG ONLY: it cannot decide gate C (balanced) or gate A (noise-limited).',
    config=dict(dataset=DATASET, backbone=BACKBONE, alpha=ALPHA, n_cal=N_CAL,
                n_splits_A=N_SPLITS_A, n_splits_BC=N_SPLITS_BC, n_null=N_NULL,
                stable_threshold=STABLE_THRESHOLD,
                feature_sets={k: list(v) for k, v in FEATURE_SETS.items()},
                amendments=['protocol_amendments.md#amendment-2']),
    seed=SEED, results=results,
    conclusion='DEBUG RUN - machinery exercised; CIFAR-100 does not decide the Phase-1 gate',
    started_at=time.time())
print('report:', report)
